# Fuel Economy Analysis & Prediction (2008–2018)
## Full Capstone Project Notebook

This notebook fully addresses:
1. Data Analysis & Cleaning
2. Data Visualization & Insights
3. Data Merging, Clustering, Classification & Regression

Dataset: EPA Fuel Economy Data (2008–2018)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set()


## Load All Datasets

In [ ]:
years = list(range(2008, 2019))
dfs = []

for year in years:
    df = pd.read_csv(f"data/data_{year}.csv")
    df['Year'] = year
    dfs.append(df)

print(f"Loaded {len(dfs)} datasets")


## Problem 1: Exploratory Data Analysis

In [ ]:
for i, df in enumerate(dfs):
    print(f"\n===== Dataset {years[i]} =====")
    print("Samples:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Duplicates:", df.duplicated().sum())
    print("Data Types:\n", df.dtypes)
    print("Missing Values:\n", df.isnull().sum())
    print("Unique Values:\n", df.nunique())


### Example: Value Counts

In [ ]:
print(dfs[0]['Fuel Type'].value_counts())


## Data Cleaning

In [ ]:
cleaned_dfs = []

for df in dfs:
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    df = df.fillna(method='ffill')
    cleaned_dfs.append(df)


## Problem 3: Merge Datasets

In [ ]:
common_cols = set(cleaned_dfs[0].columns)

for df in cleaned_dfs[1:]:
    common_cols = common_cols.intersection(df.columns)

cleaned_dfs = [df[list(common_cols)] for df in cleaned_dfs]
final_df = pd.concat(cleaned_dfs, ignore_index=True)

print("Final shape:", final_df.shape)


## Problem 2: Visualization & Insights

In [ ]:
# 1. Fuel Type Distribution
final_df['fuel_type'].value_counts().plot(kind='bar', title='Fuel Type Distribution')
plt.show()

# 2. City MPG Distribution
final_df['city_mpg'].hist()
plt.title("City MPG Distribution")
plt.show()

# 3. Vehicle Class vs MPG
final_df.groupby('vehicle_class')['city_mpg'].mean().plot(kind='bar')
plt.title("Vehicle Class vs City MPG")
plt.show()

# 4. SmartWay Comparison
final_df.boxplot(column='city_mpg', by='smartway')
plt.title("SmartWay vs MPG")
plt.show()

# 5. MPG Trend Over Years
final_df.groupby('year')['city_mpg'].mean().plot()
plt.title("MPG Trend Over Years")
plt.show()


## Clustering

In [ ]:
X = final_df[['city_mpg','highway_mpg']].dropna()

inertia = []
for k in range(1,10):
    km = KMeans(n_clusters=k)
    km.fit(X)
    inertia.append(km.inertia_)

plt.plot(range(1,10), inertia)
plt.title("Elbow Method")
plt.show()

kmeans = KMeans(n_clusters=3)
final_df['cluster'] = kmeans.fit_predict(X)
print(final_df.groupby('cluster').mean())


## Classification: SmartWay Prediction

In [ ]:
df_model = final_df.dropna()

X = df_model[['city_mpg','highway_mpg']]
y = LabelEncoder().fit_transform(df_model['smartway'])

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

log_model = LogisticRegression(max_iter=1000)
tree_model = DecisionTreeClassifier()

log_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

y_pred1 = log_model.predict(X_test)
y_pred2 = tree_model.predict(X_test)

print("Logistic Accuracy:", accuracy_score(y_test, y_pred1))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred2))

print("\nLogistic Report:\n", classification_report(y_test, y_pred1))
print("\nTree Report:\n", classification_report(y_test, y_pred2))


## Regression: Predict City MPG

In [ ]:
X = df_model[['highway_mpg']]
y = df_model['city_mpg']

reg = LinearRegression()
reg.fit(X,y)

print("R2 Score:", reg.score(X,y))


## Final Conclusion
- Data cleaned and analyzed across 10 years
- Trends in MPG and fuel types identified
- Clustering revealed efficiency groups
- Classification models compared
- Regression used for MPG prediction
